# Futures data quality: IB API vs DB rolling view

Wraps `scripts/futures_data_quality.py` -- six-step diagnostic:
1. Stale price detection (consecutive zero-return runs)
2. Volume/liquidity gate (if volume available)
3. Roll date contamination (auto-detected or provided)
4. Listwise vs pairwise deletion assessment
5. Covariance matrix comparison (IB pairwise, IB listwise, DB listwise)
6. Structured recommendation

Computation is polars-based throughout. Conversion to pandas happens only
at the `pypfopt.risk_models.sample_cov` call site in Step 5 (consistent
with project CLAUDE.md convention).

Requires a live IB connection for the fetch step.

In [ ]:
import sys
import logging
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'nbs' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

import hvplot.polars   # registers .hvplot on polars DataFrames/Series
import polars as pl

from scripts.futures_data_quality import (
    fetch_ib_prices, load_db_prices, run_quality_check
)

## Parameters — edit here

In [ ]:
# Instruments: defaults to the full KNOWN_INSTRUMENTS universe.
# Uncomment and edit to narrow the universe.
# INSTRUMENTS = 'MES,MNQ,ES,NQ'

DURATION         = '3 y'    # IB historical window
INCLUDE_DB       = False    # set True to also load from the local duckdb cache
STALE_THRESHOLD  = 3        # consecutive zero-return days to flag
OUTLIER_SIGMA    = 5.0      # σ for auto roll-date detection
HALFLIFE         = 60.0     # EWM covariance halflife (days)
ROLL_DATES       = None     # dict {ticker: [date_strings]} or None

HOST      = '127.0.0.1'
PORT      = 7496
CLIENT_ID = 22

In [ ]:
# Fetch IB data.
from ib_tools.ibpysync import IBPySync
from options_bt.live.run_tsmom_rebalance import KNOWN_INSTRUMENTS, _build_instruments

instruments_spec = locals().get('INSTRUMENTS', ','.join(sorted(KNOWN_INSTRUMENTS)))
instruments = _build_instruments(instruments_spec, None, 15)

print(f'instruments_spec : {instruments_spec!r}')
print(f'instruments ({len(instruments)}):')
for instr in instruments:
    sym      = instr.get('symbol')
    ib_sym   = instr.get('ib_symbol', sym)
    sig_sym  = instr.get('signal_symbol', ib_sym)
    exchange = instr.get('exchange', 'CME')
    print(f'  {sym:8s}  ib={ib_sym}  signal={sig_sym}  exchange={exchange}')

print(f'\nConnecting to IB at {HOST}:{PORT} (clientId={CLIENT_ID}) ...')
ib = IBPySync()
ib.connect(HOST, PORT, CLIENT_ID)
print('Connected.')
try:
    ib_prices = fetch_ib_prices(ib, instruments, duration=DURATION)
except Exception:
    import traceback
    traceback.print_exc()
    raise
finally:
    ib.disconnect()
    print('Disconnected.')

fetched = [c for c in ib_prices.columns if c != 'date']
print(f'\nFetched {len(fetched)}/{len(instruments)} instruments, {len(ib_prices)} rows')
print(f'Columns: {fetched}')
ib_prices.head()

In [ ]:
# Optionally load DB prices for comparison.
db_prices = None
if INCLUDE_DB:
    tickers = [instr['symbol'] for instr in instruments]
    db_prices = load_db_prices(tickers)
    if db_prices is not None:
        print(f'DB prices loaded: {len(db_prices)} rows, columns: {db_prices.columns}')
    else:
        print('DB prices unavailable -- proceeding with IB only')

## Run quality check (Steps 1–6)

In [ ]:
results = run_quality_check(
    ib_prices, db_prices,
    roll_dates=ROLL_DATES,
    stale_run_threshold=STALE_THRESHOLD,
    outlier_sigma=OUTLIER_SIGMA,
    halflife=HALFLIFE,
    print_report=True,  # prints the full text summary as it runs
)

In [ ]:
# Unpack the key artifacts.
stale   = results['stale']
rolls   = results['rolls']
deletion = results['deletion']
cov     = results['covariance']
S_final = results['S_final']    # recommended covariance matrix (pandas)
ret_final = results['ret_final']  # corresponding returns (pandas)

## Stale price runs

In [ ]:
if stale['flagged_tickers']:
    print('Instruments with stale price runs:')
    for ticker, n_rows in sorted(stale['flagged_tickers'].items()):
        print(f'  {ticker}: {n_rows} flagged rows')
        for start, end, length in stale['runs'][ticker]:
            print(f'    {start} -> {end} ({length} consecutive zeros)')
else:
    print('No stale price runs detected.')

In [ ]:
# Stale-price timeline -- all-polars: unpivot then .hvplot directly.
stale_mask = stale['stale_mask']
tickers = [c for c in stale_mask.columns if c != 'date']
if any(stale_mask[t].sum() > 0 for t in tickers):
    stale_long = (
        stale_mask
        .with_columns(
            [pl.col(t).cast(pl.Int8) for t in tickers]
            + [pl.col('date').cast(pl.Utf8)]
        )
        .unpivot(index='date', on=tickers, variable_name='instrument', value_name='stale')
    )
    stale_long.hvplot.heatmap(
        x='date', y='instrument', C='stale',
        cmap='Reds', clim=(0, 1), colorbar=False,
        width=900, height=max(200, len(tickers) * 40),
        title=f'Stale-price mask (red = zero-return run ≥ {STALE_THRESHOLD} days)',
        rot=45,
    )
else:
    print('No stale periods to plot.')

## Auto-detected roll / anomaly dates

In [ ]:
total_auto = sum(len(v) for v in rolls['auto_detected'].values())
print(f'Auto-detected roll/anomaly dates: {total_auto} across {len(rolls["auto_detected"])} instruments')
for ticker, dates in rolls['auto_detected'].items():
    if dates:
        print(f'  {ticker}: {len(dates)} dates — {dates[:5]}{" ..." if len(dates) > 5 else ""}')

## Correlation matrix (IB listwise)

In [ ]:
if cov is not None:
    import numpy as np
    S = cov['S_ib_listwise']          # pandas from pypfopt -- convert at the plot boundary
    std = np.sqrt(np.diag(S.values))
    cm  = S / np.outer(std, std)
    cm_long = (
        pl.from_pandas(cm.reset_index().rename(columns={'index': 'instrument'}))
        .unpivot(index='instrument', variable_name='col', value_name='corr')
    )
    cm_long.hvplot.heatmap(
        x='col', y='instrument', C='corr',
        cmap='coolwarm', clim=(-1, 1), colorbar=True,
        width=500, height=450,
        title=f'Correlation matrix (IB listwise, EWM halflife={HALFLIFE}d)',
    )
else:
    print('Covariance comparison not available (pypfopt not installed).')

## Annualised vol comparison (pairwise vs listwise, IB vs DB)

In [ ]:
if cov is not None:
    print(cov['vol_comparison'].round(4).to_string())
    if cov['flagged_vol_instruments']:
        print(f'\nInstruments with >5pp vol diff between pairwise and listwise: {cov["flagged_vol_instruments"]}')

## Deletion assessment

In [ ]:
print(f'Total rows:    {deletion["total_rows"]}')
print(f'Listwise rows: {deletion["listwise_rows"]} ({deletion["listwise_pct"]}%)')
print(f'Pairwise N CV: {deletion["pairwise_n_cv"]:.2%} '
      f'({"reliable" if deletion["pairwise_reliable"] else "UNRELIABLE"})')
print(f'Recommendation: {deletion["recommended_strategy"]}')

In [ ]:
# Per-pair N heatmap -- polars DataFrame built directly from the dict.
if deletion['pairwise_n']:
    rows = []
    for (ta, tb), n in deletion['pairwise_n'].items():
        rows += [{'x': ta, 'y': tb, 'N': float(n)},
                 {'x': tb, 'y': ta, 'N': float(n)}]
    tickers_all = sorted(set(t for pair in deletion['pairwise_n'] for t in pair))
    for t in tickers_all:
        rows.append({'x': t, 'y': t, 'N': float(deletion['total_rows'])})
    pl.DataFrame(rows).hvplot.heatmap(
        x='x', y='y', C='N',
        cmap='YlOrRd_r', colorbar=True,
        width=500, height=450,
        title='Per-pair N under pairwise deletion (lower = more data loss)',
    )